# Carpet diagram — lgm temperature benchmarking

Step toward an update of IPCC AR5 WG1 Fig. 9.12 (a *portrait plot* / *carpet
diagram*). For each PMIP lgm (Last Glacial Maximum (21 ka)) simulation we:

1. Load the annual-mean near-surface temperature field `tas_spatialmean_ann` from the
   CVDP output, and the matching `piControl` run.
2. Compute the **lgm − piControl anomaly** on each model's native grid.
3. **Sample** that anomaly field at the location of every proxy reconstruction.
4. Summarise the model–data mismatch as the **root-mean-squared error (RMSE)** against
   each reconstruction compilation (Bartlein 21 ka (Bartlein et al. 2011), P2F (Past 2 Future, in prep),
                      Cleator (Cleator et al. 2020) and Osman (LGMR; Osman et al. 2021)).

Intermediate tables are written to `output/`. Run with the `my-cli-py` conda env.

In [ ]:
import os, glob, re
import numpy as np
import pandas as pd
import xarray as xr

ROOT      = os.getcwd()  # run the notebook from the carpet_diagram/ directory
CVDP_DIR  = os.path.join(ROOT, 'cvdp_output_by_experiment')
EXPERIMENT = 'lgm'
RECON_DIR = os.path.join(ROOT, 'recons', EXPERIMENT)
OUT_DIR   = os.path.join(ROOT, 'output')
os.makedirs(OUT_DIR, exist_ok=True)

VAR        = 'tas_spatialmean_ann'
print('experiment:', EXPERIMENT)
print('CVDP   :', CVDP_DIR)
print('recons :', RECON_DIR)
print('output :', OUT_DIR)

## Step 1 — Reconstruction compilations

Four LGM (21 ka) compilations. Two are scattered proxy sites, with unit weights:

- **Bartlein** — the Bartlein et al. (2011) pollen-based mean-annual-temperature synthesis
  at 21 ka (`Bartlein_mat_21ka.csv`, the NOAA `mat_delta_21ka_ALL_grid_2x2.csv`), the 21 ka
  counterpart of the file used for the midHolocene row. 98 cells with an anomaly, in the
  `mat_anm_mean` column. Note this is the proxy compilation that Cleator assimilated, so
  the Cleator row below is not independent of it.
- **P2F** — the Past 2 Future ocean-temperature compilation (Cutmore et al., Durham,
  in prep; work in progress), `P2F-Recommended-SST-records_LGM_C.Brierley.xlsx`. Only rows
  whose `Recommendation` column reads exactly `Recommend` are kept; the anomaly is
  `Annual SST` minus the site's `ERSSTv6 mean annual SST (1870-1899)` modern climatology.

The other two are near-global gridded products, so each point gets a `cos(latitude)`
weight for an area-fair RMSE:

- **Cleator** — the Cleator et al. (2020) vegetation-model-inversion **data assimilation**
  product (its own metadata: "made by combining pollen based reconstructions (from Bartlein
  et al. 2011) and averaged outputs of LGM simulations from … PMIP … under a variational
  data assimilation technique"). A long metadata preamble precedes a `#`-prefixed header row
  (line 77); the only relevant field is `MAT` (mean-annual-temperature anomaly, °C).
- **Osman** — the Last Glacial Maximum Reanalysis (Osman et al. 2021), the ensemble-mean
  `sat` field from `Osman_LGMR_21ka_SAT_anom_climo.nc`, already differenced to a 21 ka −
  (0–1 ka) anomaly. The 96×144 grid is flattened to points, dropping fill values.

In [ ]:
# --- Bartlein: the 21 ka pollen MAT synthesis (scattered 2x2 cells) ---
bar = pd.read_csv(os.path.join(RECON_DIR, 'Bartlein_mat_21ka.csv'))
bar.columns = [c.strip() for c in bar.columns]   # headers carry leading spaces
bartlein = pd.DataFrame({
    'compilation': 'Bartlein', 'reference': 'Bartlein et al. 2011', 'site': np.nan,
    'Proxy': 'pollen MAT',
    'Latitude': bar['lat'].astype(float), 'Longitude': bar['lon'].astype(float),
    'Anom': bar['mat_anm_mean'].astype(float),
})
bartlein['source_table'] = 'Bartlein_mat_21ka.csv'
bartlein['weight'] = 1.0                  # scattered sites: equal weight

# --- P2F: Past 2 Future SST compilation (Cutmore et al., in prep) ---
P2F_FILE = 'P2F-Recommended-SST-records_LGM_C.Brierley.xlsx'
p2f = pd.read_excel(os.path.join(RECON_DIR, P2F_FILE))
p2f = p2f[p2f['Recommendation'] == 'Recommend'].copy()
p2f_df = pd.DataFrame({
    'compilation': 'P2F', 'reference': 'Past 2 Future (Cutmore et al., in prep)',
    'site': p2f['Core'].astype(str).str.strip(),
    'Proxy': p2f['Proxy/Proxies'],
    'Latitude': p2f['Latitude'].astype(float), 'Longitude': p2f['Longitude'].astype(float),
    'Anom': (p2f['Annual SST'] - p2f['ERSSTv6 mean annual SST (1870-1899)']).astype(float),
})
p2f_df['source_table'] = P2F_FILE
p2f_df['weight'] = 1.0                    # scattered sites: equal weight

# --- Cleator: gridded land MAT anomalies from a CSV with a preamble ---
CLE_COLS = ['lat', 'lon', 'MI', 'MAP', 'MAT', 'MTCO', 'MTWA', 'GDD5',
            'MI_SD', 'MAP_SD', 'MAT_SD', 'MTCO_SD', 'MTWA_SD', 'GDD5_SD']
cle = pd.read_csv(os.path.join(RECON_DIR, 'cleator2020_recon.csv'),
                  skiprows=77, header=None, names=CLE_COLS)
cle = cle.apply(pd.to_numeric, errors='coerce').dropna(subset=['lat', 'lon', 'MAT'])
cleator = pd.DataFrame({
    'compilation': 'Cleator', 'reference': 'Cleator et al. 2020', 'site': np.nan,
    'Proxy': 'pollen MAT (assim.)',
    'Latitude': cle['lat'].astype(float), 'Longitude': cle['lon'].astype(float),
    'Anom': cle['MAT'].astype(float),
})
cleator['source_table'] = 'cleator2020_recon.csv'

# --- Osman: flatten the gridded LGMR SAT anomaly field to points ---
ds = xr.open_dataset(os.path.join(RECON_DIR, 'Osman_LGMR_21ka_SAT_anom_climo.nc'),
                     decode_times=False)
sat = ds['sat'].where(np.abs(ds['sat']) < 1e30)
lon2d, lat2d = np.meshgrid(sat['lon'].values, sat['lat'].values)
osman = pd.DataFrame({
    'compilation': 'Osman', 'reference': 'Osman et al. 2021', 'site': np.nan,
    'Proxy': 'LGMR SAT (assim.)',
    'Latitude': lat2d.ravel().astype(float), 'Longitude': lon2d.ravel().astype(float),
    'Anom': sat.values.ravel().astype(float),
})
osman['source_table'] = 'Osman_LGMR_21ka_SAT_anom_climo.nc'

for gridded in (cleator, osman):
    # Regular lat/lon grids: cos-latitude weight makes the RMSE area-fair.
    gridded['weight'] = np.cos(np.deg2rad(gridded['Latitude']))

recon = pd.concat([bartlein, p2f_df, cleator, osman], ignore_index=True)
recon = recon.dropna(subset=['Latitude', 'Longitude', 'Anom'])

print(recon.groupby('compilation').size())
recon_out = os.path.join(OUT_DIR, f'recon_points_{EXPERIMENT}.csv')
recon.to_csv(recon_out, index=False)
print('wrote', recon_out)
recon.head()

## Step 2 — Model lgm − piControl anomalies

Each model writes its CVDP field on its own native grid, so anomalies are computed
per model. We pair every `lgm` file with the same model's `piControl` file
(the main file is the one named `<model>_<experiment>.cvdp_data.<years>.nc`; auxiliary
per-variable files such as `.siconc.` / `.zos.` / `.monsoon.` / `.tas.indices.` carry a
extra token before the years and are skipped). If the two grids ever differ, the control
is bilinearly regridded onto the lgm grid before differencing.

In [ ]:
def model_files(experiment):
    """Map model name -> path for the main CVDP file of an experiment.

    Only `<model>_<experiment>.cvdp_data.<start>-<end>.nc` is the main file; the
    auxiliary per-variable outputs (`.siconc.`, `.zos.`, `.monsoon.`, `.tas.indices.`)
    put an extra token before the year range and hold no `tas_spatialmean_ann`.
    """
    out = {}
    pat = os.path.join(CVDP_DIR, experiment, f'*_{experiment}.cvdp_data.*.nc')
    main = re.compile(rf'^(?P<model>.+)_{re.escape(experiment)}\.cvdp_data\.\d+-\d+\.nc$')
    for f in sorted(glob.glob(pat)):
        m = main.match(os.path.basename(f))
        if m:
            out[m.group('model')] = f
    return out

def has_var(path):
    """True if the CVDP file actually carries the tas field (a few don't)."""
    with xr.open_dataset(path, decode_times=False) as ds:
        return VAR in ds.variables

exp_files = model_files(EXPERIMENT)
pi_files  = model_files('piControl')
paired = sorted(set(exp_files) & set(pi_files))
# Some CVDP files lack tas_spatialmean_ann entirely — drop those models.
models = [m for m in paired if has_var(exp_files[m]) and has_var(pi_files[m])]
print(f'{len(models)} models with both {EXPERIMENT} and piControl (and a tas field):')
print(models)
missing_pi = sorted(set(exp_files) - set(pi_files))
if missing_pi:
    print(f'{EXPERIMENT} models with no piControl (skipped):', missing_pi)
no_var = [m for m in paired if m not in models]
if no_var:
    print(f'models dropped — no {VAR} in CVDP file:', no_var)

In [ ]:
def load_field(path):
    da = xr.open_dataset(path, decode_times=False)[VAR].sortby('lat').sortby('lon')
    # A few CVDP grids (e.g. LOVECLIM piControl) carry duplicate lon values,
    # which break interpolation; keep the first occurrence of each coordinate.
    for dim in ('lat', 'lon'):
        _, idx = np.unique(da[dim].values, return_index=True)
        if len(idx) != da.sizes[dim]:
            da = da.isel({dim: np.sort(idx)})
    return da

anomalies = {}
for m in models:
    exp_field = load_field(exp_files[m])
    pi  = load_field(pi_files[m])
    if exp_field.shape != pi.shape or not (np.allclose(exp_field.lat, pi.lat) and np.allclose(exp_field.lon, pi.lon)):
        pi = pi.interp(lat=exp_field.lat, lon=exp_field.lon)
    anomalies[m] = (exp_field - pi).rename('tas_anom')
    print(f'{m:18s} grid {exp_field.shape}  mean anom {float(anomalies[m].mean()):+.2f} C')

## Step 3 — Sample model anomalies at reconstruction locations

Model longitudes run 0–360°, the proxy longitudes −180–180°, so targets are wrapped to
0–360 and the field is made cyclic in longitude before bilinear interpolation.

Each recon point carries a `weight` used later in the RMSE. Scattered-site compilations
weight every point equally (1.0); gridded near-global reconstructions (Cleator, Osman,
Erb, Tierney) set `weight = cos(latitude)` so the RMSE is area-fair rather than
pole-heavy.

In [ ]:
def sample_points(field, lats, lons):
    """Bilinearly sample a (lat, lon) field at scattered points; lon made cyclic."""
    lon_cyc = np.append(field.lon.values, field.lon.values[0] + 360.0)
    fcyc = xr.concat([field, field.isel(lon=0)], dim='lon').assign_coords(lon=lon_cyc)
    ta = xr.DataArray(np.asarray(lats), dims='point')
    to = xr.DataArray(np.asarray(lons) % 360.0, dims='point')
    return fcyc.interp(lat=ta, lon=to).values

sampled = recon[['compilation', 'reference', 'site', 'Proxy',
                 'Latitude', 'Longitude', 'Anom']].copy()
sampled = sampled.rename(columns={'Anom': 'recon_anom'})
# Optional per-point weight (defaults to equal weighting when Step 1 omits it).
sampled['weight'] = recon['weight'].values if 'weight' in recon.columns else 1.0
for m in models:
    sampled[m] = sample_points(anomalies[m], sampled['Latitude'].values, sampled['Longitude'].values)

sampled_out = os.path.join(OUT_DIR, f'model_anom_at_recon_{EXPERIMENT}.csv')
sampled.to_csv(sampled_out, index=False)
print('wrote', sampled_out, '  shape', sampled.shape)
sampled.head()

## Step 4 — RMSE of each model against each compilation

For every model × compilation we take the model-minus-proxy difference across all proxy
points in that compilation and report the (weight-weighted) RMSE and bias, plus the number
of points contributing (a point off the model grid edge can return NaN). With unit weights
this is the ordinary RMSE; gridded compilations use `cos(latitude)` weights (Step 3).

In [ ]:
records = []
for m in models:
    for comp, grp in sampled.groupby('compilation'):
        diff = grp[m].values - grp['recon_anom'].values
        w = grp['weight'].values
        valid = np.isfinite(diff) & np.isfinite(w)
        n = int(valid.sum())
        if n:
            dv, wv = diff[valid], w[valid]
            rmse = float(np.sqrt(np.sum(wv * dv ** 2) / np.sum(wv)))
            bias = float(np.sum(wv * dv) / np.sum(wv))
        else:
            rmse = bias = np.nan
        records.append({'model': m, 'compilation': comp, 'n_points': n,
                        'rmse': rmse, 'bias': bias})

rmse_long = pd.DataFrame(records)
rmse_wide = rmse_long.pivot(index='model', columns='compilation', values='rmse')
rmse_wide.columns = [f'{c}_RMSE' for c in rmse_wide.columns]

rmse_long.to_csv(os.path.join(OUT_DIR, f'rmse_long_{EXPERIMENT}.csv'), index=False)
rmse_wide.to_csv(os.path.join(OUT_DIR, f'rmse_summary_{EXPERIMENT}.csv'))
print(f'wrote rmse_long_{EXPERIMENT}.csv and rmse_summary_{EXPERIMENT}.csv')
rmse_wide.sort_values(rmse_wide.columns[0])

These RMSE values are the building blocks of the carpet diagram: one column per model,
one row per (period, reconstruction compilation), coloured by RMSE. `carpet_figure.py`
reads every `output/rmse_long_<period>.csv`, so re-running this notebook for a new period
makes its rows appear in the portrait plot automatically.